# 22DM015 Final Project — Financial PhraseBank
## Person A: Part 1 — Setting Up the Problem (1.5 pts)

**Dataset:** `takala/financial_phrasebank`, config `sentences_allagree` (2,264 sentences).
**Labels:** 0 = negative, 1 = neutral, 2 = positive.

### Shared data contract (set by Person D — do NOT re-split)
- Splits are committed under `data/` as CSVs: `train` (1584), `val` (227), `test` (453), `labeled_32` (32).
- The **32 labelled** examples are a balanced sample from train (11 neg / 10 neu / 11 pos).
- Part 2 'unlabelled' data = train minus the 32 (`unlabeled_pool`).
- Evaluate headline numbers on **`test`** only; tune on **`val`**. Use `eval_utils.evaluate` so we're comparable.
- Log every result with `eval_utils.log_result(...)` into `results/results.csv`.

> **AI disclosure:** any AI-generated code/text in this notebook must be watermarked and declared (course rule). Interpretation, methodology justification, and analysis must be student-authored.

In [ ]:
# --- Reproducibility seed (required by the assignment) ---
import os, random, sys
import numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Make the shared helpers importable (they live in the repo root, one level up).
sys.path.append(os.path.abspath('..'))
import data_utils as du
import eval_utils as eu

splits = du.load_splits()            # identical data for everyone
train, val, test = splits['train'], splits['val'], splits['test']
labeled_32 = splits['labeled_32']
pool = du.unlabeled_pool(splits)     # Part 2 'unlabelled' data
PERSON = 'A'
for k, v in splits.items():
    print(f'{k:11s} {len(v):5d}', dict(v['label_name'].value_counts()))

## 1a. Bibliography & State of the Art (0.25)
_Student-authored markdown._ Task = 3-class financial sentiment on Financial PhraseBank (Malo et al., 2014). Note SOTA references (e.g. FinBERT, Araci 2019) and benchmarks. Keep AI out of the written analysis.

## 1b. Dataset Description (0.5)
Size, class distribution, peculiarities (neutral-heavy ~61%; short single-sentence headlines). Add basic descriptive stats below.

In [ ]:
# Descriptive stats: class balance, sentence length distribution, etc.
import pandas as pd
full = pd.concat([train, val, test], ignore_index=True)
print(full['label_name'].value_counts(normalize=True).round(3))
full['n_words'] = full['text'].str.split().map(len)
print(full['n_words'].describe())
# TODO: plot class distribution and length histogram

## 1c. Random Classifier Performance (0.25)
Expected performance of a random classifier, *with an implementation*. Two natural baselines: uniform (1/3 each) and prior-weighted (sample by class frequency). Report accuracy and macro-F1 on test so it's comparable to the real models.

In [ ]:
# Random baselines on the TEST set
import numpy as np
y_test = test['label'].to_numpy()
priors = train['label'].value_counts(normalize=True).sort_index().to_numpy()
rng = np.random.default_rng(SEED)
y_uniform = rng.integers(0, 3, size=len(y_test))
y_prior   = rng.choice([0,1,2], size=len(y_test), p=priors)
print('uniform  ', eu.evaluate(y_test, y_uniform))
print('prior-wt ', eu.evaluate(y_test, y_prior))
eu.log_result('random-uniform', 'baseline', 0, eu.evaluate(y_test, y_uniform), person=PERSON)
eu.log_result('random-prior',   'baseline', 0, eu.evaluate(y_test, y_prior),   person=PERSON)

## 1d. Rule-Based Baseline (0.5)
Build a lexicon/rule classifier (e.g. count positive vs negative finance terms; default neutral). Discuss its performance vs dataset complexity and (if available) human agreement levels.

In [ ]:
# Minimal lexicon rule-based classifier — extend the word lists.
POS = {'increase','rose','growth','profit','up','gain','higher','improved','strong'}
NEG = {'decrease','fell','loss','down','decline','lower','weak','cut','drop'}
def rule_predict(text):
    t = text.lower().split()
    p, n = sum(w in POS for w in t), sum(w in NEG for w in t)
    if p > n: return 2
    if n > p: return 0
    return 1  # default neutral
y_pred = test['text'].map(rule_predict).to_numpy()
m = eu.evaluate(y_test, y_pred); print(m)
eu.log_result('rule-based', 'baseline', 0, m, person=PERSON, notes='lexicon')